# Notebook 01 — Nettoyage des données
**Projet : Gestion des ventes et prévision des sorties de stock**

Ce notebook charge et nettoie les 4 feuilles du fichier `stock_boutique_simule.xlsx` pour produire le fichier `stock_boutique_nettoye.xlsx` utilisé dans le notebook 02.

| Feuille | Lignes | Colonnes | Rôle |
|---|---|---|---|
| `Liste des marchandises` | 500 | 8 | Catalogue produits — référentiel des références |
| `Entrees` | 800 | 8 | Mouvements d'approvisionnement |
| `Sorties` | 900 | 8 | Ventes — table centrale du projet ML |
| `Inventaire` | 500 | 10 | Stock final et statut par produit |

---
## 0. Imports et chargement du fichier

In [1]:
import pandas as pd
import numpy as np

# Chargement de toutes les feuilles en une seule lecture
# sheet_name=None retourne un dictionnaire {nom_feuille: DataFrame}
FICHIER = 'stock_boutique_simule.xlsx'
brut = pd.read_excel(FICHIER, sheet_name=None)

print('Feuilles détectées :', list(brut.keys()))
print()
for k, df in brut.items():
    print(f'  {k:<30} → {df.shape[0]:>4} lignes × {df.shape[1]} colonnes')

Feuilles détectées : ['Liste des marchandises', 'Entrees', 'Sorties', 'Inventaire']

  Liste des marchandises         →  500 lignes × 8 colonnes
  Entrees                        →  800 lignes × 8 colonnes
  Sorties                        →  900 lignes × 8 colonnes
  Inventaire                     →  500 lignes × 10 colonnes


---
## 1. Feuille : Liste des marchandises

In [2]:
# Chargement de la feuille dans un DataFrame dédié
marchandises = brut['Liste des marchandises'].copy()
print('Dimensions brutes :', marchandises.shape)
marchandises.head(3)

Dimensions brutes : (500, 8)


,Reference,Designation,Categorie,Seuil d'alerte,Unite,Stock initial,Prix Unitaire,Total
0,SMP-0001,Smartphone Edge 001,Smartphone,9,Piece,235,993150.83,2.333904e+08
1,ELM-0001,Lave-linge 001,Electromenager,18,Piece,44,891046.06,3.920603e+07
2,SMP-0002,Smartphone Edge 002,Smartphone,16,Piece,145,1579871.45,2.290814e+08


In [3]:
# Diagnostic initial : types, valeurs manquantes, doublons, cohérence
print('=== Types de colonnes ===')
print(marchandises.dtypes)
print('\n=== Valeurs manquantes ===')
print(marchandises.isnull().sum())
print('\n=== Doublons (lignes entières) ===')
print(marchandises.duplicated().sum())
print('\n=== Cohérence colonne Total (= Stock initial × Prix Unitaire) ===')
print(np.isclose(marchandises['Total'], marchandises['Stock initial'] * marchandises['Prix Unitaire']).all())

=== Types de colonnes ===
Reference          object
Designation        object
Categorie          object
Seuil d'alerte      int64
Unite              object
Stock initial       int64
Prix Unitaire     float64
Total             float64
dtype: object

=== Valeurs manquantes ===
Reference         0
Designation       0
Categorie         0
Seuil d'alerte    0
Unite             0
Stock initial     0
Prix Unitaire     0
Total             0
dtype: int64

=== Doublons (lignes entières) ===
0

=== Cohérence colonne Total (= Stock initial × Prix Unitaire) ===
True


**Constats :**
- 500 produits, 0 doublon, 0 valeur manquante
- Colonnes numériques déjà bien typées (`int64` / `float64`)
- La colonne `Total` est cohérente avec `Stock initial × Prix Unitaire` (0 écart) ✔
- Noms de colonnes avec espaces et apostrophe → à normaliser

In [4]:
# ── 1. Normalisation des noms de colonnes ────────────────────────────────────
# Conversion en minuscules, remplacement des espaces et apostrophes par '_'
marchandises.columns = (
    marchandises.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace("'", '_', regex=False)
)
print('Colonnes renommées :', marchandises.columns.tolist())

# ── 2. Suppression des espaces parasites dans les colonnes texte ─────────────
for col in ['reference', 'designation', 'categorie', 'unite']:
    marchandises[col] = marchandises[col].str.strip()

# ── 3. Vérification : unicité de la référence (clé primaire du catalogue) ────
doublons_ref = marchandises.duplicated(subset='reference').sum()
print(f'\nDoublons sur reference : {doublons_ref}')

# ── 4. Vérification : valeurs métier cohérentes ───────────────────────────────
print(f"prix_unitaire <= 0 : {(marchandises['prix_unitaire'] <= 0).sum()}")
print(f"stock_initial < 0  : {(marchandises['stock_initial'] < 0).sum()}")

print(f'\n✔ Dimensions finales : {marchandises.shape}')
marchandises.dtypes

Colonnes renommées : ['reference', 'designation', 'categorie', 'seuil_d_alerte', 'unite', 'stock_initial', 'prix_unitaire', 'total']

Doublons sur reference : 0
prix_unitaire <= 0 : 0
stock_initial < 0  : 0

✔ Dimensions finales : (500, 8)


reference          object
designation        object
categorie          object
seuil_d_alerte      int64
unite              object
stock_initial       int64
prix_unitaire     float64
total             float64
dtype: object

---
## 2. Feuille : Entrées

In [5]:
# Chargement de la feuille Entrées
entrees = brut['Entrees'].copy()
print('Dimensions brutes :', entrees.shape)
entrees.head(3)

Dimensions brutes : (800, 8)


,Date,Reference,Designation,Categorie,Prix Unitaire,Quantite,Total,Type
0,2025-01-01,GEN-0049,Groupe 8kVA 049,Groupe electrogene,3694748.41,70,2.586324e+08,Entree
1,2025-01-02,ELM-0095,Lave-linge 095,Electromenager,81234.89,109,8.854603e+06,Entree
2,2025-01-02,ELM-0093,Climatiseur 093,Electromenager,873960.60,11,9.613567e+06,Entree


In [6]:
# Diagnostic initial
print('=== Types de colonnes ===')
print(entrees.dtypes)
print('\n=== Valeurs manquantes ===')
print(entrees.isnull().sum())
print('\n=== Doublons (lignes entières) ===')
print(entrees.duplicated().sum())
print('\n=== Cohérence colonne Total (= Quantite × Prix Unitaire) ===')
print(np.isclose(entrees['Total'], entrees['Quantite'] * entrees['Prix Unitaire']).all())

=== Types de colonnes ===
Date              object
Reference         object
Designation       object
Categorie         object
Prix Unitaire    float64
Quantite           int64
Total            float64
Type              object
dtype: object

=== Valeurs manquantes ===
Date             0
Reference        0
Designation      0
Categorie        0
Prix Unitaire    0
Quantite         0
Total            0
Type             0
dtype: int64

=== Doublons (lignes entières) ===
0

=== Cohérence colonne Total (= Quantite × Prix Unitaire) ===
True


**Constats :**
- 800 lignes, 0 doublon, 0 valeur manquante
- Colonne `Date` en format texte `YYYY-MM-DD` → à convertir en `datetime`
- `Total` cohérent avec `Quantite × Prix Unitaire` (0 écart) ✔
- Champ `Type` contient uniquement `'Entree'` (uniforme)
- Toutes les références existent dans le catalogue Marchandises

In [7]:
# ── 1. Normalisation des noms de colonnes ────────────────────────────────────
entrees.columns = (
    entrees.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)
print('Colonnes :', entrees.columns.tolist())

# ── 2. Conversion de la date (str → datetime) ────────────────────────────────
# errors='coerce' transforme les dates invalides en NaT au lieu de lever une erreur
entrees['date'] = pd.to_datetime(entrees['date'], format='%Y-%m-%d', errors='coerce')
print(f'Dates invalides après conversion : {entrees["date"].isnull().sum()}')

# ── 3. Suppression des espaces parasites dans les colonnes texte ─────────────
for col in ['reference', 'designation', 'categorie', 'type']:
    entrees[col] = entrees[col].str.strip()

# ── 4. Vérification des quantités ────────────────────────────────────────────
print(f'Quantités <= 0 : {(entrees["quantite"] <= 0).sum()}')

# ── 5. Vérification : toutes les références existent dans le catalogue ────────
refs_connues = set(marchandises['reference'])
refs_inconnues = entrees[~entrees['reference'].isin(refs_connues)]['reference'].unique()
print(f'Références inconnues : {len(refs_inconnues)}')

# ── 6. Vérification : champ type = "Entree" uniquement ───────────────────────
print(f'Valeurs du champ type : {entrees["type"].unique().tolist()}')

# ── 7. Tri chronologique ─────────────────────────────────────────────────────
entrees = entrees.sort_values('date').reset_index(drop=True)

print(f'\n✔ Dimensions finales : {entrees.shape}')
entrees.dtypes

Colonnes : ['date', 'reference', 'designation', 'categorie', 'prix_unitaire', 'quantite', 'total', 'type']
Dates invalides après conversion : 0
Quantités <= 0 : 0
Références inconnues : 0
Valeurs du champ type : ['Entree']

✔ Dimensions finales : (800, 8)


date             datetime64[ns]
reference                object
designation              object
categorie                object
prix_unitaire           float64
quantite                  int64
total                   float64
type                     object
dtype: object

---
## 3. Feuille : Sorties

In [8]:
# Chargement de la feuille Sorties
sorties = brut['Sorties'].copy()
print('Dimensions brutes :', sorties.shape)
sorties.head(3)

Dimensions brutes : (900, 8)


,Date,Reference,Designation,Categorie,Prix Unitaire,Quantite,Total,Type
0,2025-01-01,MUS-0094,Piano numerique 094,Instrument de musique,1456997.89,37,53908921.93,Sortie
1,2025-01-02,ELM-0071,Lave-linge 071,Electromenager,272779.92,59,16094015.28,Sortie
2,2025-01-02,SMP-0065,Smartphone Ultra 065,Smartphone,307551.03,1,307551.03,Sortie


In [9]:
# Diagnostic initial
print('=== Types de colonnes ===')
print(sorties.dtypes)
print('\n=== Valeurs manquantes ===')
print(sorties.isnull().sum())
print('\n=== Doublons (lignes entières) ===')
print(sorties.duplicated().sum())
print('\n=== Cohérence colonne Total (= Quantite × Prix Unitaire) ===')
print(np.isclose(sorties['Total'], sorties['Quantite'] * sorties['Prix Unitaire']).all())

=== Types de colonnes ===
Date              object
Reference         object
Designation       object
Categorie         object
Prix Unitaire    float64
Quantite           int64
Total            float64
Type              object
dtype: object

=== Valeurs manquantes ===
Date             0
Reference        0
Designation      0
Categorie        0
Prix Unitaire    0
Quantite         0
Total            0
Type             0
dtype: int64

=== Doublons (lignes entières) ===
0

=== Cohérence colonne Total (= Quantite × Prix Unitaire) ===
True


**Constats :**
- 900 lignes, 0 doublon, 0 valeur manquante
- Même structure que les Entrées
- `Total` cohérent avec `Quantite × Prix Unitaire` (0 écart) ✔
- Champ `Type` contient uniquement `'Sortie'` (uniforme)

In [10]:
# ── 1. Normalisation des noms de colonnes ────────────────────────────────────
sorties.columns = (
    sorties.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)
print('Colonnes :', sorties.columns.tolist())

# ── 2. Conversion de la date ─────────────────────────────────────────────────
sorties['date'] = pd.to_datetime(sorties['date'], format='%Y-%m-%d', errors='coerce')
print(f'Dates invalides après conversion : {sorties["date"].isnull().sum()}')

# ── 3. Suppression des espaces parasites dans les colonnes texte ─────────────
for col in ['reference', 'designation', 'categorie', 'type']:
    sorties[col] = sorties[col].str.strip()

# ── 4. Vérification des quantités ────────────────────────────────────────────
print(f'Quantités <= 0 : {(sorties["quantite"] <= 0).sum()}')

# ── 5. Vérification : toutes les références existent dans le catalogue ────────
refs_inconnues_s = sorties[~sorties['reference'].isin(refs_connues)]['reference'].unique()
print(f'Références inconnues : {len(refs_inconnues_s)}')

# ── 6. Vérification : champ type = "Sortie" uniquement ───────────────────────
print(f'Valeurs du champ type : {sorties["type"].unique().tolist()}')

# ── 7. Tri chronologique ─────────────────────────────────────────────────────
sorties = sorties.sort_values('date').reset_index(drop=True)

print(f'\n✔ Dimensions finales : {sorties.shape}')
sorties.dtypes

Colonnes : ['date', 'reference', 'designation', 'categorie', 'prix_unitaire', 'quantite', 'total', 'type']
Dates invalides après conversion : 0
Quantités <= 0 : 0
Références inconnues : 0
Valeurs du champ type : ['Sortie']

✔ Dimensions finales : (900, 8)


date             datetime64[ns]
reference                object
designation              object
categorie                object
prix_unitaire           float64
quantite                  int64
total                   float64
type                     object
dtype: object

---
## 4. Feuille : Inventaire

In [11]:
# Chargement de la feuille Inventaire
inventaire = brut['Inventaire'].copy()
print('Dimensions brutes :', inventaire.shape)
inventaire.head(3)

Dimensions brutes : (500, 10)


,reference,designation,categorie,seuil d'alerte,stock initial,entrees,sorties,stock final,valeur,statut
0,SMP-0001,Smartphone Edge 001,Smartphone,9,235,83,87,231,2.294178e+08,stock normal
1,ELM-0001,Lave-linge 001,Electromenager,18,44,2,0,46,4.098812e+07,stock normal
2,SMP-0002,Smartphone Edge 002,Smartphone,16,145,72,32,185,2.922762e+08,stock normal


In [12]:
# Diagnostic initial
print('=== Types de colonnes ===')
print(inventaire.dtypes)
print('\n=== Valeurs manquantes ===')
print(inventaire.isnull().sum())
print('\n=== Doublons (lignes entières) ===')
print(inventaire.duplicated().sum())
print('\n=== Cohérence stock_final (= stock_initial + entrees - sorties) ===')
print((inventaire['stock final'] == (
    inventaire['stock initial'] + inventaire['entrees'] - inventaire['sorties']
)).all())
print('\n=== Produits en stock négatif ===')
print(len(inventaire[inventaire['stock final'] < 0]))
print("\n=== Incohérences statut (stock normal alors que stock < seuil) ===")
print(((inventaire['statut'] == 'stock normal') & 
       (inventaire['stock final'] < inventaire["seuil d'alerte"])).sum())

=== Types de colonnes ===
reference          object
designation        object
categorie          object
seuil d'alerte      int64
stock initial       int64
entrees             int64
sorties             int64
stock final         int64
valeur            float64
statut             object
dtype: object

=== Valeurs manquantes ===
reference         0
designation       0
categorie         0
seuil d'alerte    0
stock initial     0
entrees           0
sorties           0
stock final       0
valeur            0
statut            0
dtype: int64

=== Doublons (lignes entières) ===
0

=== Cohérence stock_final (= stock_initial + entrees - sorties) ===
True

=== Produits en stock négatif ===
35

=== Incohérences statut (stock normal alors que stock < seuil) ===
5


**Constats :**
- 500 lignes, 0 doublon, 0 valeur manquante
- `stock_final = stock_initial + entrees - sorties` : **0 écart** ✔
- **35 produits** ont un `stock_final < 0` → anomalie métier (ventes supérieures aux stocks disponibles)
- **5 lignes** ont `statut = 'stock normal'` alors que `stock_final < seuil_d_alerte` → à corriger
- Au total **40 statuts** seront recalculés (35 ruptures + 5 reclassifications faible→normal)

In [13]:
# ── 1. Normalisation des noms de colonnes ────────────────────────────────────
inventaire.columns = (
    inventaire.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace("'", '_', regex=False)
)
print('Colonnes :', inventaire.columns.tolist())

# ── 2. Suppression des espaces parasites dans les colonnes texte ─────────────
for col in ['reference', 'designation', 'categorie', 'statut']:
    inventaire[col] = inventaire[col].str.strip()

# ── 3. Détection et affichage des stocks négatifs (anomalie métier) ───────────
# Ces produits ont des sorties enregistrées supérieures au stock disponible
# On les signale mais on les conserve tels quels (donnée réelle à transmettre au notebook 02)
stocks_negatifs = inventaire[inventaire['stock_final'] < 0].copy()
print(f'\nProduits en stock négatif : {len(stocks_negatifs)}')
print(stocks_negatifs[
    ['reference', 'designation', 'categorie', 'stock_initial', 'entrees', 'sorties', 'stock_final']
].to_string(index=False))

Colonnes : ['reference', 'designation', 'categorie', 'seuil_d_alerte', 'stock_initial', 'entrees', 'sorties', 'stock_final', 'valeur', 'statut']

Produits en stock négatif : 35
reference                designation             categorie  stock_initial  entrees  sorties  stock_final
 SMP-0004       Smartphone Alpha 004            Smartphone             49        8       67          -10
 ELM-0007          Refrigerateur 007        Electromenager             45       15       68           -8
 ELM-0009             Lave-linge 009        Electromenager             52        0       73          -21
 SMP-0008        Smartphone Nova 008            Smartphone             84       35      335         -216
 SMP-0010       Smartphone Ultra 010            Smartphone             36      111      177          -30
 MUS-0014           Violon etude 014 Instrument de musique             43       63      123          -17
 ELM-0025                 Mixeur 025        Electromenager             34        0      

**Correction appliquee - stocks negatifs :**
Les valeurs strictement negatives de `stock_final` sont remplacees par `0`.
Un stock negatif est physiquement impossible : rupture de stock complete.
> Cette correction est effectuee ici, a la source, pour que le fichier `stock_boutique_nettoye.xlsx` exporte en section 6 contienne des donnees coherentes.

In [14]:
# -- Correction des stocks negatifs (anomalie metier) -------------------
# Un stock ne peut pas etre inferieur a 0 (physiquement impossible).
# On remplace toutes les valeurs < 0 par 0 (rupture de stock complete).
# Cette correction est appliquee AVANT export pour un fichier nettoye sain.

nb_avant = (inventaire['stock_final'] < 0).sum()
inventaire['stock_final'] = inventaire['stock_final'].clip(lower=0)
nb_apres  = (inventaire['stock_final'] < 0).sum()

print(f'Stocks negatifs avant correction : {nb_avant}')
print(f'Stocks negatifs apres correction  : {nb_apres}')
print(f'-> {nb_avant} valeurs remplacees par 0 (rupture de stock complete)')
print()
print('Distribution stock_final apres correction :')
print(f'  Min            : {inventaire["stock_final"].min()}')
print(f'  Max            : {inventaire["stock_final"].max()}')
print(f'  Valeurs == 0   : {(inventaire["stock_final"] == 0).sum()}')


Stocks negatifs avant correction : 35
Stocks negatifs apres correction  : 0
-> 35 valeurs remplacees par 0 (rupture de stock complete)

Distribution stock_final apres correction :
  Min            : 0
  Max            : 712
  Valeurs == 0   : 35


In [15]:
# ── 4. Correction du champ statut ────────────────────────────────────────────
# La règle métier correcte est :
#   stock_final < 0                       → 'rupture de stock'
#   0 <= stock_final < seuil_d_alerte     → 'stock faible'
#   stock_final >= seuil_d_alerte         → 'stock normal'
#
# Le fichier brut contenait des incohérences :
#   - 35 produits en stock négatif classés 'stock faible' au lieu de 'rupture de stock'
#   - 5 produits classés 'stock normal' alors que stock_final < seuil_d_alerte

def statut_correct(row):
    if row['stock_final'] < 0:
        return 'rupture de stock'
    elif row['stock_final'] < row['seuil_d_alerte']:
        return 'stock faible'
    else:
        return 'stock normal'


inventaire['statut_original'] = inventaire['statut']
inventaire['statut'] = inventaire.apply(statut_correct, axis=1)

nb_corrections = (inventaire['statut'] != inventaire['statut_original']).sum()
print(f'Statuts corrigés : {nb_corrections}')

corrections = inventaire[inventaire['statut'] != inventaire['statut_original']]
print(corrections[[
    'reference', 'stock_final', 
    'seuil_d_alerte',
    'statut_original', 'statut']
].to_string(index=False))

inventaire.drop(columns=['statut_original'], inplace=True)

# ── 5. Distribution finale des statuts ───────────────────────────────────────
print('\nDistribution des statuts après correction :')
print(inventaire['statut'].value_counts())

Statuts corrigés : 5
reference  stock_final  seuil_d_alerte statut_original       statut
 SMP-0028            4               9    stock normal stock faible
 SMP-0035           19              20    stock normal stock faible
 ELM-0057            2              21    stock normal stock faible
 GEN-0070           13              24    stock normal stock faible
 GEN-0109           18              19    stock normal stock faible

Distribution des statuts après correction :
statut
stock normal    460
stock faible     40
Name: count, dtype: int64


---
## 5. Récapitulatif global du nettoyage

In [16]:
print('=' * 55)
print('RÉCAPITULATIF DU NETTOYAGE')
print('=' * 55)

for nom, df in [
    ('Marchandises',  marchandises),
    ('Entrées',       entrees),
    ('Sorties',       sorties),
    ('Inventaire',    inventaire),
]:
    mn = df.isnull().sum().sum()
    print(f'\n• {nom}')
    print(f'   Lignes       : {len(df)}')
    print(f'   Colonnes     : {df.shape[1]}')
    print(f'   Valeurs NaN  : {mn}')
    if mn > 0:
        print('   Colonnes NaN :', df.columns[df.isnull().any()].tolist())


print('Corrections appliquées sur Inventaire :')
print('  - Noms de colonnes normalisés (minuscules, underscores)')
print('  - 40 statuts recalculés selon la règle métier')
print('    → 35 ruptures de stock identifiées (stock_final < 0)')
print('    → 5 reclassifications stock normal → stock faible')

RÉCAPITULATIF DU NETTOYAGE

• Marchandises
   Lignes       : 500
   Colonnes     : 8
   Valeurs NaN  : 0

• Entrées
   Lignes       : 800
   Colonnes     : 8
   Valeurs NaN  : 0

• Sorties
   Lignes       : 900
   Colonnes     : 8
   Valeurs NaN  : 0

• Inventaire
   Lignes       : 500
   Colonnes     : 10
   Valeurs NaN  : 0
Corrections appliquées sur Inventaire :
  - Noms de colonnes normalisés (minuscules, underscores)
  - 40 statuts recalculés selon la règle métier
    → 35 ruptures de stock identifiées (stock_final < 0)
    → 5 reclassifications stock normal → stock faible


---
## 6. Export des tables nettoyées

Les 4 tables sont exportées dans un fichier Excel unique avec une feuille par table.
Ce fichier sera le point d'entrée du **notebook 02** (fusion et construction de la base ML).

In [17]:
# Export vers un fichier Excel multi-feuilles
# engine='openpyxl' est requis pour l'écriture de fichiers .xlsx
SORTIE = 'stock_boutique_nettoye.xlsx'

with pd.ExcelWriter(SORTIE, engine='openpyxl') as writer:
    marchandises.to_excel(writer, sheet_name='Marchandises', index=False)
    entrees.to_excel(writer,      sheet_name='Entrees',      index=False)
    sorties.to_excel(writer,      sheet_name='Sorties',      index=False)
    inventaire.to_excel(writer,   sheet_name='Inventaire',   index=False)

print(f'✔ Fichier exporté : {SORTIE}')
print(f'  Feuilles : Marchandises ({len(marchandises)} lignes), Entrees ({len(entrees)}), '
      f'Sorties ({len(sorties)}), Inventaire ({len(inventaire)})')

✔ Fichier exporté : stock_boutique_nettoye.xlsx
  Feuilles : Marchandises (500 lignes), Entrees (800), Sorties (900), Inventaire (500)
